# 🧠 Agent Planning Debugger — Making AI Reasoning Transparent
**A Step-by-Step Trace Viewer for an AutonomousPlannerAgent**

_Week 8 Community Exercise · LiteLLM · OpenAI Function Calling · Gradio_

---

## What I Learned in Week 8

Week 8 culminated in a multi-agent deal-hunting system. The hardest part to understand was the **AutonomousPlannerAgent** — it's a black box that somehow selects tools, reasons about them, and chains calls together. This notebook tears that box open:

- **Day 1** — SpecialistAgent: wrapping a function as a callable tool
- **Day 3** — ScannerAgent: chaining agents together (one agent calls another)
- **Day 4** — AutonomousPlannerAgent: LLM selects which tool to call via function-calling API

## Why This Project

The `DealAgentFramework` works seamlessly — but *why* does the planner pick one tool over another? I wanted to:
1. Rebuild a **minimal but complete** planner pattern locally (no Modal required)
2. Capture every tool call as a **structured trace** `{step, tool_name, input, reasoning, output}`
3. Render that trace in Gradio as a **numbered timeline** so the reasoning chain becomes teachable

---

## Components

| Component | Purpose |
|---|---|
| `litellm` + function-calling | LLM selects which tool to invoke (mirrors Week 8 planner pattern) |
| 3 standalone tools | `zero_shot_price`, `rag_price`, `ensemble_price` |
| Trace list `list[dict]` | Captured per-step reasoning with input/output |
| `gr.Blocks`, `gr.Markdown`, `gr.JSON` | Timeline view, Final Answer tab, Raw JSON inspector |

---

## How It Works

1. **Input** — User types a product description
2. **Plan** — LLM receives the tools as JSON schemas; picks which to call and why
3. **Execute** — Each tool call is intercepted, logged to the trace, and its output fed back
4. **Loop** — Planner continues until it emits a final price (no more tool calls)
5. **Display** — Gradio renders: Timeline · Final Answer · Raw Messages JSON

**Hardware:** Runs fully locally on CPU. Uses `openai/gpt-4.1-mini` by default — switch to `ollama/llama3.2` for a free run (Ollama must be running).

---

> **Note:** This notebook does NOT require Modal.com. It reimplements the planner pattern as a self-contained local script using the OpenAI function-calling API via `litellm`.

## Cell 1 — Install Dependencies

In [8]:
# Run once — restart kernel after installation if needed
!uv pip install -q litellm openai gradio python-dotenv

print("✅ Dependencies installed.")

✅ Dependencies installed.


## Cell 2 — Imports & Configuration

In [9]:
import os
import re
import json
import random
from dotenv import load_dotenv

load_dotenv(override=True)

# ── Model config ─────────────────────────────────────────────────────────────
#
#  "gpt"       → openai/gpt-4.1-mini  (requires OPENAI_API_KEY)
#  "anthropic" → anthropic/claude-3-5-haiku (requires ANTHROPIC_API_KEY)
#  "ollama"    → ollama/llama3.2  (requires Ollama running locally, free)
#
MODEL_CHOICE = "gpt"  # ← change this

MODELS = {
    "gpt"       : "openai/gpt-4.1-mini",
    "anthropic" : "anthropic/claude-3-5-haiku-20241022",
    "ollama"    : "ollama/llama3.2",
}
OLLAMA_BASE  = "http://localhost:11434"
MODEL_ID     = MODELS[MODEL_CHOICE]

MAX_STEPS    = 5   # max tool-call rounds before forcing a final answer

# Print key status
openai_key    = os.getenv("OPENAI_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

print(f"MODEL_CHOICE : {MODEL_CHOICE} → {MODEL_ID}")
print(f"OpenAI key   : {'✅' if openai_key    else '⚠️  not set'}")
print(f"Anthropic key: {'✅' if anthropic_key else '⚠️  not set'}")
print(f"Ollama       : select MODEL_CHOICE='ollama' + ensure 'ollama serve' is running")
print("\n✅ Configuration ready.")

MODEL_CHOICE : gpt → openai/gpt-4.1-mini
OpenAI key   : ✅
Anthropic key: ✅
Ollama       : select MODEL_CHOICE='ollama' + ensure 'ollama serve' is running

✅ Configuration ready.


## Cell 3 — Define 3 Agent Tools

In [10]:
from litellm import completion as litellm_completion

# ── Tool 1: Zero-shot price estimate via LLM ────────────────────────────────
def zero_shot_price(product_description: str) -> str:
    """
    Estimate the price of an Amazon product using a frontier LLM zero-shot.
    Fast — but no context about comparable products.
    """
    kwargs = dict(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": "Estimate this product's Amazon price. Reply with only a dollar amount like $49.99."},
            {"role": "user",   "content": product_description},
        ],
        max_tokens=15,
    )
    if MODEL_CHOICE == "ollama":
        kwargs["api_base"] = OLLAMA_BASE
    reply = litellm_completion(**kwargs).choices[0].message.content.strip()
    return f"Zero-shot estimate: {reply}"


# ── Tool 2: RAG-style price estimate using similar product heuristics ────────
# Simplified: instead of real ChromaDB lookups, we use a curated reference table.
# This mimics the FrontierAgent RAG pattern from Week 8 Day 2.

_REFERENCE_PRODUCTS = [
    {"description": "Wireless Bluetooth noise-cancelling headphones, 30hr battery", "price": 79.99},
    {"description": "USB-C hub with HDMI, 4K, 7 ports, aluminum", "price": 39.99},
    {"description": "Mechanical keyboard, TKL, Cherry MX Red switches, RGB backlight", "price": 89.99},
    {"description": "Laptop stand, adjustable height, aluminium, foldable", "price": 29.99},
    {"description": "1080p webcam, autofocus, dual microphone, plug and play", "price": 59.99},
    {"description": "Smart Wi-Fi plug, voice control, energy monitoring, 2-pack", "price": 19.99},
    {"description": "Electric standing desk, motorised, 3 memory settings, 55 inch", "price": 399.00},
    {"description": "AA rechargeable batteries 8-pack with charger, 2800mAh NiMH", "price": 24.99},
    {"description": "Robot vacuum cleaner, self-emptying, laser navigation, works with Alexa", "price": 349.99},
    {"description": "Air purifier, HEPA filter, covers 500 sqft, quiet mode", "price": 89.99},
    {"description": "USB condenser microphone for streaming, PC Mac, RGB, shock mount", "price": 129.99},
    {"description": "Professional XLR microphone for podcast and studio, cardioid", "price": 99.99},
]

RAG_SIMILARITY_THRESHOLD = 2

def rag_price(product_description: str) -> str:
    """
    Retrieve 3 similar reference products and ask the LLM to price based on context.
    If no reference has enough keyword overlap (similarity < threshold), fall back to zero-shot.
    """
    query_words = set(product_description.lower().split())
    scored = sorted(
        _REFERENCE_PRODUCTS,
        key=lambda p: len(query_words & set(p["description"].lower().split())),
        reverse=True
    )
    top3 = scored[:3]
    best_overlap = len(query_words & set(top3[0]["description"].lower().split()))

    if best_overlap < RAG_SIMILARITY_THRESHOLD:
        fallback = zero_shot_price(product_description)
        price_part = fallback.replace("Zero-shot estimate: ", "").strip()
        return f"RAG: no similar products in reference set; zero-shot fallback: {price_part}"

    context = "Reference products for context:\n"
    for ref in top3:
        context += f"  - {ref['description']} → ${ref['price']:.2f}\n"
    ref_avg = sum(r["price"] for r in top3) / len(top3)
    context += f"\nAverage of these reference prices: ${ref_avg:.2f}"

    user_msg = f"{context}\n\nProduct to price: {product_description}\n\nYour estimate MUST be grounded in the reference prices above (close to or between them). Reply with only a dollar amount."

    kwargs = dict(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": "You are a pricing expert. Your estimate MUST be based on the reference product prices provided; do not ignore them. If the product is similar to a reference, your answer should be near that price or between the listed reference prices."},
            {"role": "user",   "content": user_msg},
        ],
        max_tokens=20,
    )
    if MODEL_CHOICE == "ollama":
        kwargs["api_base"] = OLLAMA_BASE
    reply = litellm_completion(**kwargs).choices[0].message.content.strip()
    return f"RAG-based estimate (using {len(top3)} similar products): {reply}"


# ── Tool 3: Ensemble — average of zero-shot + rule-based heuristic ───────────
def ensemble_price(product_description: str) -> str:
    """
    Combine a zero-shot LLM estimate with a simple length/keyword heuristic.
    Mirrors the Week 8 Day 2 EnsembleAgent blending pattern.
    """
    # Zero-shot component
    kwargs = dict(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": "Estimate this product's price. Reply with only a number (no $ sign)."},
            {"role": "user",   "content": product_description},
        ],
        max_tokens=10,
    )
    if MODEL_CHOICE == "ollama":
        kwargs["api_base"] = OLLAMA_BASE
    llm_reply = litellm_completion(**kwargs).choices[0].message.content.strip()
    llm_price = float(re.search(r"[\d.]+", llm_reply).group()) if re.search(r"[\d.]+", llm_reply) else 50.0

    # Heuristic component: word count × $1.5, clipped to [10, 500]
    word_count    = len(product_description.split())
    heuristic_price = max(10.0, min(500.0, word_count * 1.5))

    ensemble = round(0.85 * llm_price + 0.15 * heuristic_price, 2)
    return (f"Ensemble estimate: ${ensemble:.2f}  "
            f"(LLM=${llm_price:.2f} × 0.85 + heuristic=${heuristic_price:.2f} × 0.15)")


# ── Tool registry ────────────────────────────────────────────────────────────
TOOL_FUNCTIONS = {
    "zero_shot_price" : zero_shot_price,
    "rag_price"       : rag_price,
    "ensemble_price"  : ensemble_price,
}

print("✅ 3 agent tools defined:", list(TOOL_FUNCTIONS.keys()))

✅ 3 agent tools defined: ['zero_shot_price', 'rag_price', 'ensemble_price']


## Cell 4 — JSON Schema for Function Calling

In [11]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "zero_shot_price",
            "description": (
                "Estimate the Amazon retail price of a product using a frontier LLM with no additional context. "
                "Fast and general. Best for common, well-known product categories."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "product_description": {
                        "type": "string",
                        "description": "The full product description to price."
                    }
                },
                "required": ["product_description"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rag_price",
            "description": (
                "Retrieve 3 similar reference products from a product database, then estimate the price with "
                "that context. More accurate than zero-shot for niche or technical products."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "product_description": {
                        "type": "string",
                        "description": "The full product description to price."
                    }
                },
                "required": ["product_description"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "ensemble_price",
            "description": (
                "Combine a zero-shot LLM estimate with a structural heuristic (word count/length). "
                "Best when you want a robust estimate that is less sensitive to prompt phrasing."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "product_description": {
                        "type": "string",
                        "description": "The full product description to price."
                    }
                },
                "required": ["product_description"]
            }
        }
    }
]

PLANNER_SYSTEM = """You are an autonomous pricing agent. Your job is to estimate the retail price of Amazon products.

You have 3 tools available:
- zero_shot_price: fast, no context — use for common products
- rag_price: retrieves similar products for context — use for niche/technical items
- ensemble_price: blends LLM + heuristic — use when robustness matters

Strategy (you MUST use at least 2 tools before giving your final answer):
1. First get a quick estimate with zero_shot_price.
2. Then use rag_price for comparable products (or ensemble_price for robustness).
3. Only after you have at least two estimates, synthesize them and report: FINAL PRICE: $XX.XX

Use at most 3 tool calls. Always end with FINAL PRICE: $XX.XX"""

print("✅ Tool schemas and planner system prompt ready.")

✅ Tool schemas and planner system prompt ready.


## Cell 5 — The Planner: Tool Execution Loop with Trace Capture

In [12]:
def run_planner(product_description: str, mode: str = "autonomous", progress_callback=None) -> tuple[list[dict], str, list]:
    """
    Run the planner on a product description.
    mode: "autonomous" (LLM picks tools) or "pipeline" (fixed sequence: zero_shot → rag → ensemble → synthesize).
    progress_callback: optional fn(fraction, desc) for UI progress (e.g. Gradio progress).
    Returns:
      trace       : list of step dicts {step, tool_name, tool_input, tool_output, note}
      final_price : the extracted final price string
      messages    : raw message list (for JSON tab)
    """
    def report(pct: float, desc: str):
        if progress_callback:
            progress_callback(pct, desc=desc)

    if mode == "pipeline":
        trace = []
        steps = [
            ("zero_shot_price", zero_shot_price),
            ("rag_price", rag_price),
            ("ensemble_price", ensemble_price),
        ]
        for step, (fn_name, fn) in enumerate(steps, start=1):
            report(0.15 + 0.2 * (step - 1), f"Calling {fn_name}...")
            try:
                out = fn(product_description)
                note = "✓ Called successfully"
            except Exception as e:
                out = f"Error: {e}"
                note = "⚠️ Tool raised an exception"
            trace.append({
                "step": step, "tool_name": fn_name,
                "tool_input": product_description[:200],
                "tool_output": out, "note": note,
            })
        report(0.75, "Synthesizing final price...")
        context = "\n".join(f"- {t['tool_name']}: {t['tool_output']}" for t in trace)
        synth_prompt = f"Given these three price estimates for \"{product_description[:100]}...\":\n\n{context}\n\nSynthesize and report a single final estimate as: FINAL PRICE: $XX.XX"
        kwargs = dict(model=MODEL_ID, messages=[
            {"role": "system", "content": "You are a pricing agent. Output only FINAL PRICE: $XX.XX"},
            {"role": "user", "content": synth_prompt},
        ], max_tokens=50)
        if MODEL_CHOICE == "ollama":
            kwargs["api_base"] = OLLAMA_BASE
        content = litellm_completion(**kwargs).choices[0].message.content or ""
        price_match = re.search(r"FINAL PRICE[:\s]+\$?([\d.,]+)", content, re.I)
        final_price = f"${price_match.group(1)}" if price_match else (f"${re.findall(r'[\d.]+', content)[-1]}" if re.findall(r"[\d.]+", content) else "Unknown")
        trace.append({"step": len(trace) + 1, "tool_name": "— (final answer)", "tool_input": "", "tool_output": content, "note": f"Planner emitted final answer: {final_price}"})
        messages = [{"role": "user", "content": product_description}, {"role": "assistant", "content": content}]
        return trace, final_price, messages

    messages = [
        {"role": "system", "content": PLANNER_SYSTEM},
        {"role": "user",   "content": f"Please estimate the price of this product:\n\n{product_description}"},
    ]
    trace       = []
    final_price = "Unknown"
    step        = 0

    for round_num in range(MAX_STEPS):
        report(0.1 + 0.7 * (round_num / max(MAX_STEPS, 1)), "Planning (LLM)...")
        kwargs = dict(
            model=MODEL_ID,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            max_tokens=500,
        )
        if MODEL_CHOICE == "ollama":
            kwargs["api_base"] = OLLAMA_BASE

        response    = litellm_completion(**kwargs)
        msg         = response.choices[0].message
        finish      = response.choices[0].finish_reason

        # Append assistant message to history
        messages.append(msg.model_dump() if hasattr(msg, "model_dump") else dict(msg))

        # ── No tool call → planner gave a final answer ────────────────────────
        if not msg.tool_calls:
            content = msg.content or ""
            price_match = re.search(r"FINAL PRICE[:\s]+\$?([\d.,]+)", content, re.I)
            if price_match:
                final_price = f"${price_match.group(1)}"
            else:
                # Try to grab any dollar-like number
                nums = re.findall(r"\$?([\d]{1,4}(?:\.\d{1,2})?)", content)
                final_price = f"${nums[-1]}" if nums else content[:60]

            trace.append({
                "step"        : step + 1,
                "tool_name"   : "— (final answer)",
                "tool_input"  : "",
                "tool_output" : content,
                "note"        : f"Planner emitted final answer: {final_price}"
            })
            break

        for tool_call in msg.tool_calls:
            step      += 1
            fn_name   = tool_call.function.name
            fn_args   = json.loads(tool_call.function.arguments)
            tool_input = fn_args.get("product_description", product_description)
            report(0.1 + 0.7 * ((round_num + step / 3) / MAX_STEPS), f"Calling {fn_name}...")
            try:
                tool_output = TOOL_FUNCTIONS[fn_name](tool_input)
                error_note  = ""
            except Exception as e:
                tool_output = f"Error: {e}"
                error_note  = "⚠️ Tool raised an exception"

            trace.append({
                "step"        : step,
                "tool_name"   : fn_name,
                "tool_input"  : tool_input[:200],
                "tool_output" : tool_output,
                "note"        : error_note or f"✓ Called successfully",
            })

            # Feed tool result back to the planner
            messages.append({
                "role"         : "tool",
                "tool_call_id" : tool_call.id,
                "name"         : fn_name,
                "content"      : tool_output,
            })

    return trace, final_price, messages


# Quick smoke test
print("Smoke test: running planner on 'USB-C laptop charger 65W'...")
test_trace, test_price, _ = run_planner("USB-C laptop charger 65W with foldable plug and 6ft cable")
print(f"Final price: {test_price}")
print(f"Steps taken: {len(test_trace)}")
print("\n✅ Planner working correctly.")

Smoke test: running planner on 'USB-C laptop charger 65W'...
Final price: $34.99
Steps taken: 3

✅ Planner working correctly.


## Cell 6 — Format Trace for Gradio Display

In [13]:
TOOL_ICONS = {
    "zero_shot_price" : "⚡",
    "rag_price"       : "🔍",
    "ensemble_price"  : "🔀",
    "— (final answer)": "🏁",
}

def format_trace_markdown(trace: list[dict]) -> str:
    """Render the trace as a readable numbered timeline in Markdown."""
    if not trace:
        return "_No trace available._"

    lines = ["### 🔍 Agent Reasoning Trace\n"]
    for step in trace:
        icon = TOOL_ICONS.get(step["tool_name"], "🔧")
        lines.append(f"**Step {step['step']}: {icon} `{step['tool_name']}`**")
        if step["tool_input"]:
            lines.append(f"- **Input:** _{step['tool_input'][:120]}{'…' if len(step['tool_input']) > 120 else ''}_")
        lines.append(f"- **Output:** {step['tool_output']}")
        lines.append(f"- **Note:** {step['note']}")
        lines.append("")   # blank line between steps

    return "\n".join(lines)


# Preview
print(format_trace_markdown(test_trace))

### 🔍 Agent Reasoning Trace

**Step 1: ⚡ `zero_shot_price`**
- **Input:** _USB-C laptop charger 65W with foldable plug and 6ft cable_
- **Output:** Zero-shot estimate: $29.99
- **Note:** ✓ Called successfully

**Step 2: 🔍 `rag_price`**
- **Input:** _USB-C laptop charger 65W with foldable plug and 6ft cable_
- **Output:** RAG-based estimate (using 3 similar products): 39.99
- **Note:** ✓ Called successfully

**Step 3: 🏁 `— (final answer)`**
- **Output:** The zero-shot price estimate for the USB-C laptop charger 65W with foldable plug and 6ft cable is $29.99. The RAG-based estimate, using 3 similar products for context, is $39.99. 

Synthesizing these two, the product likely has a price in the range of $30 to $40.

FINAL PRICE: $34.99
- **Note:** Planner emitted final answer: $34.99



## Cell 7 — Gradio UI: Timeline · Final Answer · Raw JSON

In [14]:
import gradio as gr

def run_agent_ui(product_description: str, mode: str, progress=gr.Progress()):
    """Gradio handler: runs planner and returns trace, price, and raw messages."""
    if not product_description.strip():
        return "⚠️ Please enter a product description.", "", "{}"

    progress(0.05, desc="Starting agent...")
    try:
        trace, final_price, messages = run_planner(product_description, mode=mode, progress_callback=progress)
    except Exception as e:
        return f"❌ Error: {e}", "", "{}"

    progress(0.9, desc="Formatting trace...")

    trace_md  = format_trace_markdown(trace)
    answer_md = (
        f"## 💰 Final Price Estimate\n\n"
        f"**{final_price}**\n\n"
        f"> Agent used **{len([s for s in trace if s['tool_name'] != '— (final answer)'])}** tool call(s) "
        f"across **{len(trace)}** steps to reach this conclusion."
    )
    raw_json  = json.dumps(messages, indent=2, default=str)

    progress(1.0, desc="Done!")
    return trace_md, answer_md, raw_json


EXAMPLE_PRODUCTS = [
    "HyperX Quadcast S RGB USB Condenser Microphone for PC, PS4, PS5 and Mac, Anti-Vibration Shock Mount",
    "Aeron Chair by Herman Miller, Fully Adjustable with PostureFit SL, High Performance Gaming and Work Chair",
    "LEGO Technic Bugatti Chiron 42083 Building Kit with Bugatti Model Car",
]

with gr.Blocks(title="Agent Planning Debugger") as demo:

    gr.Markdown(
        "# 🧠 Agent Planning Debugger\n"
        "**Transparent step-by-step reasoning for an AutonomousPlannerAgent**\n\n"
        f"_Model: `{MODEL_ID}` · Tools: zero_shot_price, rag_price, ensemble_price_"
    )

    with gr.Row():
        product_input = gr.Textbox(
            label="Product Description",
            placeholder="e.g. Sony WH-1000XM5 Wireless Noise Cancelling Headphones",
            lines=3,
            scale=4,
        )
        mode_dropdown = gr.Dropdown(
            choices=["autonomous", "pipeline"],
            value="autonomous",
            label="Mode",
            info="Autonomous: LLM picks tools. Pipeline: fixed sequence (zero_shot → rag → ensemble → synthesize).",
        )
        run_btn = gr.Button("🚀 Run Agent", variant="primary", scale=1, size="lg")

    gr.Examples(
        examples=[[p] for p in EXAMPLE_PRODUCTS],
        inputs=product_input,
        label="Try an example product",
    )

    with gr.Tabs():
        with gr.Tab("🔍 Trace Timeline"):
            gr.Markdown(
                "_Each numbered step shows which tool the agent chose, what it passed in, "
                "what it got back, and why._"
            )
            trace_output = gr.Markdown()

        with gr.Tab("💰 Final Answer"):
            answer_output = gr.Markdown()

        with gr.Tab("📄 Raw JSON Messages"):
            gr.Markdown(
                "_The exact `messages` list sent to and received from the LLM. "
                "This is what the AutonomousPlannerAgent's loop looks like under the hood._"
            )
            json_output = gr.Code(language="json", label="messages")

    run_btn.click(
        run_agent_ui,
        inputs=[product_input, mode_dropdown],
        outputs=[trace_output, answer_output, json_output],
    )

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


---

## Concepts Demonstrated

| Cell | Concept | What it shows |
|---|---|---|
| 3 | Agent tools as plain Python functions | Wrapping logic as a callable — mirrors Week 8 Day 1 SpecialistAgent |
| 4 | OpenAI function-calling JSON schema | The exact format the LLM uses to select a tool |
| 5 | Agentic loop | Message history grows per step; loop ends when LLM emits no tool calls |
| 5 | Trace capture | Each tool call logged as a structured dict — makes reasoning inspectable |
| 6 | Trace formatting | Converting raw dicts into a human-readable Markdown timeline |
| 7 | Gradio `gr.Code` (JSON tab) | Displaying the raw messages array — great for debugging |
| 7 | `gr.Progress` | Feedback during the agent loop (mirrors Week 4 HF progress bar) |

---

## Extensions Possible

1. **Add your Week 8 SpecialistAgent as Tool 4** — deploy on Modal and call `pricer.price.remote()`
2. **Connect real ChromaDB** — replace the `_REFERENCE_PRODUCTS` list in `rag_price` with a real vector store
3. **Multi-turn conversation** — let the user follow up with "what about a refurbished version?" and watch the agent re-plan
4. **Planning prompt editor** — expose `PLANNER_SYSTEM` as a Gradio `gr.Textbox` so users can tweak the strategy
5. **Export trace to CSV** — add a download button for the trace log to compare runs across products
